> ⚠️ **This notebook cannot be run from this repository.**
>
> It depends on the [Charsiu](https://github.com/lingjzhu/charsiu) source tree
> (`sys.path.append('src/')` → `from Charsiu import charsiu_forced_aligner`),
> which is **not vendored here**. Only the notebook is kept in this repo, as a
> record of how the Charsiu baseline numbers in the paper were produced.
>
> To reproduce:
>
> 1. Clone Charsiu separately and set up its own environment
>    (it pins older `transformers` / `torch` versions and will conflict with
>    this project's dependencies — do **not** install it here).
>    ```bash
>    git clone https://github.com/lingjzhu/charsiu.git
>    cd charsiu
>    # create an isolated env, then install Charsiu's requirements
>    ```
> 2. Copy this notebook into the Charsiu repository root (next to `src/`).
> 3. Fix the corpus paths in the notebook (`MFA_ROOT`, `WRD_ROOT`,
>    `BUCKEYE_ROOT`) to match your machine.
> 4. Run it from inside that environment.
>
> The pretrained checkpoint (`charsiu/en_w2v2_fc_10ms`) is downloaded from the
> Hugging Face Hub on first run. The numbers reported in the paper were obtained
> on 2026-08-26; upstream updates to the checkpoint or to Charsiu's inference
> code may change the results.

In [16]:
import sys
sys.path.append('src/')
from pathlib import Path
import numpy as np
import soundfile as sf
from tqdm import tqdm
from Charsiu import charsiu_forced_aligner

In [17]:
c = charsiu_forced_aligner(aligner='charsiu/en_w2v2_fc_10ms')

In [ ]:
def read_ref(wrd_path, sr=16000):
    ref = []
    for line in open(wrd_path):
        line = line.strip()
        if not line:
            continue
        s, e, w = line.split(maxsplit=2)
        ref.append((int(s) / sr, int(e) / sr, w.strip().lower()))
    return ref


def lcs_match(ref_words, hyp_words):
    n, m = len(ref_words), len(hyp_words)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n):
        for j in range(m):
            if ref_words[i] == hyp_words[j]:
                dp[i + 1][j + 1] = dp[i][j] + 1
            else:
                dp[i + 1][j + 1] = max(dp[i][j + 1], dp[i + 1][j])
    pairs = []
    i, j = n, m
    while i > 0 and j > 0:
        if ref_words[i - 1] == hyp_words[j - 1]:
            pairs.append((i - 1, j - 1))
            i -= 1
            j -= 1
        elif dp[i - 1][j] >= dp[i][j - 1]:
            i -= 1
        else:
            j -= 1
    pairs.reverse()
    return pairs


def read_text(txt_path):
    line = open(txt_path).read().strip()
    parts = line.split(maxsplit=2)
    if len(parts) == 3 and parts[0].isdigit() and parts[1].isdigit():
        return parts[2]
    return line


def report(name, errs):
    e = np.array(errs)
    print(f"{name:12s}  n={len(e):7d}  WBE={e.mean()*1000:6.2f} ms  "
          + f"P10={100*(e<=.010).mean():5.2f}  P25={100*(e<=.025).mean():5.2f}  "
          + f"P50={100*(e<=.050).mean():5.2f}  P100={100*(e<=.100).mean():5.2f}")


def evaluate(root, wav_glob="*.wav"):
    root = Path(root)
    wavs = sorted(root.rglob(wav_glob))
    print(f"{len(wavs)} wavs in {root}")

    all_errs, all_starts, all_ends = [], [], []
    n_error = 0
    total_bounds = kept_bounds = 0

    for wav in tqdm(wavs):
        wrd = wav.with_suffix(".WRD")
        if not wrd.exists():
            wrd = wav.with_suffix(".wrd")
        txt = wav.with_suffix(".TXT")
        if not txt.exists():
            txt = wav.with_suffix(".txt")
        if not (wrd.exists() and txt.exists()):
            n_error += 1
            continue

        ref = read_ref(wrd)
        total_bounds += 2 * len(ref)

        audio, sr = sf.read(wav)
        try:
            _, pred_words = c.align(audio=audio, text=read_text(txt))
        except Exception:
            n_error += 1
            continue

        hyp = [(s, e, w.strip().lower()) for s, e, w in pred_words
               if w and w.strip() and not w.strip().startswith('[')]

        pairs = lcs_match([w for *_, w in ref], [w for *_, w in hyp])

        for ri, hi in pairs:
            rs, re_, _ = ref[ri]
            hs, he, _ = hyp[hi]
            all_starts.append(abs(hs - rs))
            all_ends.append(abs(he - re_))
            all_errs.extend([abs(hs - rs), abs(he - re_)])
            kept_bounds += 2

    print(f"\nerror={n_error}")
    print(f"boundary coverage = {100*kept_bounds/total_bounds:.2f}%  "
          + f"({kept_bounds}/{total_bounds})\n")
    report("start+end", all_errs)
    report("start only", all_starts)
    report("end only", all_ends)

In [19]:
print("=== TIMIT ===")
evaluate("/shared/data_zfs/blue2959/TIMIT/TEST", "*.WAV")

=== TIMIT ===
1680 wavs in /shared/data_zfs/blue2959/TIMIT/TEST


  0%|          | 0/1680 [00:00<?, ?it/s]

100%|██████████| 1680/1680 [00:22<00:00, 74.26it/s] 


error=0
boundary coverage = 99.41%  (28934/29106)

start+end     n=  28934  WBE= 25.10 ms  P10=35.51  P25=66.78  P50=88.34  P100=97.24
start only    n=  14467  WBE= 26.90 ms  P10=33.28  P25=62.83  P50=87.57  P100=97.39
end only      n=  14467  WBE= 23.31 ms  P10=37.73  P25=70.73  P50=89.11  P100=97.09


In [20]:
print("\n=== Buckeye ===")
evaluate("/shared/data_zfs/blue2959/Buckeye-grid", "*.wav")


=== Buckeye ===
19273 wavs in /shared/data_zfs/blue2959/Buckeye-grid


100%|██████████| 19273/19273 [04:52<00:00, 65.94it/s] 


error=0
boundary coverage = 99.38%  (437144/439858)

start+end     n= 437144  WBE= 29.22 ms  P10=34.99  P25=68.99  P50=87.40  P100=95.44
start only    n= 218572  WBE= 31.46 ms  P10=33.38  P25=66.60  P50=86.76  P100=95.16
end only      n= 218572  WBE= 26.97 ms  P10=36.59  P25=71.38  P50=88.04  P100=95.71
